Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [1]:
NO_RECORDS = 15

In [2]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, enrich_subjects_from_openaire

d:\eosc_dc\metadata-warehouse\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [4]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")

if host == '':
    host = "127.0.0.1"

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [5]:
connection_params

{'user': 'admin',
 'password': 'test',
 'dbname': 'datasetdb',
 'port': 5432,
 'host': '127.0.0.1'}

In [6]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [7]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '8409b435-a7a8-4527-a955-436ad8d92b85::38b31716-8dbc-4c90-8aff-2b92e923a499', 'title': 'Situation professionnelle deux ans après des licenciements collectifs dans le secteur industriel - 2011', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:oai="http://www.openarchives.org/OAI/2.0/" xmlns:datacite="http://datacite.org/schema/kernel-4">\n  <header>\n    <identifier>oai:swissubase.ch:38b31716-8dbc-4c90-8aff-2b92e923a499</identifier>\n    <datestamp>2026-03-24T12:04:32Z</datestamp>\n  </header>\n  <metadata>\n    <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 https://schema.datacite.org/meta/kernel-4.6/metadata.xsd">\n      <datacite:identifier identifierType="DOI">10.23662/FORS-DS-675-3</datacite:identifier>\n      <datacite:creators>\n        <datacite:creator>\n          <datacite:creatorName nameType="Personal">Baumann, Isabel</datac

In [8]:
record_titles

['Situation professionnelle deux ans après des licenciements collectifs dans le secteur industriel - 2011',
 'Clinical relevance of verbal memory impairment in polydrug Ecstasy users: A pooled data analysis',
 'Sozialisationsprozesse beim Übergang in den Lehrbetrieb (SoLe) - 2011-2017',
 'Survey among Swiss Abroad',
 'Dataset_Commons',
 'Umfrage: Immigrantinnen und Immigranten in der Schweiz-2013',
 'Widely Covering Data Collections/Database']

In [9]:
record_raw_metadata[2]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:oai="http://www.openarchives.org/OAI/2.0/" xmlns:datacite="http://datacite.org/schema/kernel-4">\n  <header>\n    <identifier>oai:swissubase.ch:dcb4c8fc-f69e-49d2-a7cb-48b9f7c007fa</identifier>\n    <datestamp>2026-03-24T12:04:32Z</datestamp>\n  </header>\n  <metadata>\n    <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 https://schema.datacite.org/meta/kernel-4.6/metadata.xsd">\n      <datacite:identifier identifierType="DOI">10.23662/FORS-DS-686-1</datacite:identifier>\n      <datacite:creators>\n        <datacite:creator>\n          <datacite:creatorName nameType="Personal">Neuenschwander, Markus P.</datacite:creatorName>\n        </datacite:creator>\n      </datacite:creators>\n      <datacite:titles>\n        <datacite:title xml:lang="de">Sozialisationsprozesse beim Übergang in den Lehrbetrieb (SoLe) - 2011-

In [10]:
get_doi(record_raw_metadata[2])

'10.23662/FORS-DS-686-1'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [11]:
doi = get_doi(record_raw_metadata[0]) 

if doi == []:
    print("NO DOI")
else:
    print(enrich_subjects_from_openaire(doi))

DEBUG:    2026-06-01 13:52:31 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-06-01 13:52:31 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.23662%2FFORS-DS-675-3 HTTP/1.1" 200 None
['8. Economic growth']


In [12]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = get_doi(record_raw_metadata[i])
    if doi != []:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 4, 5, 6]


In [13]:
id_to_enrich = [record_ids[i] for i in ix_has_doi]
doi_to_enrich = [get_doi(record_raw_metadata[i]) for i in ix_has_doi]

print(id_to_enrich)
print(doi_to_enrich)

['8409b435-a7a8-4527-a955-436ad8d92b85::38b31716-8dbc-4c90-8aff-2b92e923a499', '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/VLN5PZ', '8409b435-a7a8-4527-a955-436ad8d92b85::dcb4c8fc-f69e-49d2-a7cb-48b9f7c007fa', '8409b435-a7a8-4527-a955-436ad8d92b85::6e4b1efa-9e7d-43bb-86e5-8ffcfbca885d', '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/JESW3L', '8409b435-a7a8-4527-a955-436ad8d92b85::3557f168-dfca-4bc3-91b4-8d5ea39f902b', '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/7LUSNC']
['10.23662/FORS-DS-675-3', '10.34894/VLN5PZ', '10.23662/FORS-DS-686-1', '10.23662/FORS-DS-690-1', '10.34894/JESW3L', '10.23662/FORS-DS-692-3', '10.34894/7LUSNC']


In [14]:
from utils.enrichment_methods import _openaire_batch_doi_resolve, _opeanaire_get_doi_subjects, _ids_with_openaire_subjects

api_result = _openaire_batch_doi_resolve(doi_to_enrich)

print(api_result[:3])

DEBUG:    2026-06-01 13:52:31 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-06-01 13:52:32 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.23662%2FFORS-DS-675-3%2C10.34894%2FVLN5PZ%2C10.23662%2FFORS-DS-686-1%2C10.23662%2FFORS-DS-690-1%2C10.34894%2FJESW3L%2C10.23662%2FFORS-DS-692-3%2C10.34894%2F7LUSNC HTTP/1.1" 200 None
[{'authors': [{'fullName': 'Blatter, Joachim', 'name': 'Joachim', 'surname': 'Blatter', 'rank': 1, 'pid': None}, {'fullName': 'Schlenker, Andrea', 'name': 'Andrea', 'surname': 'Schlenker', 'rank': 2, 'pid': None}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'und', 'label': 'Undetermined'}, 'countries': None, 'subjects': None, 'mainTitle': 'Survey: Immigrants in Switzerland', 'subTitle': None, 'descriptions': None, 'publicationDate': '2018-01-01', 'publisher': None, 'embargoEndDate': None, 'sources': None, 'formats': None, 'contri

In [15]:
doi_subjects = _opeanaire_get_doi_subjects(api_result)
doi_subjects

[{'doi': '10.23662/fors-ds-686-1',
  'subjects': ['4. Education',
   '8. Economic growth',
   '10. No inequality',
   '3. Good health']},
 {'doi': '10.48573/00a2-vw63',
  'subjects': ['4. Education',
   '8. Economic growth',
   '10. No inequality',
   '3. Good health']},
 {'doi': '10.48573/c09h-h441', 'subjects': ['8. Economic growth']},
 {'doi': '10.48573/zset-sh38', 'subjects': ['8. Economic growth']},
 {'doi': '10.23662/fors-ds-675-3', 'subjects': ['8. Economic growth']},
 {'doi': '10.48573/2vz5-bh60', 'subjects': ['8. Economic growth']}]

In [16]:
id_dois = []
for ix in range(len(id_to_enrich)):
    id_dois.append(
        {
            'id': id_to_enrich[ix],
            'doi': doi_to_enrich[ix].lower(),
        }
    )
id_dois

[{'id': '8409b435-a7a8-4527-a955-436ad8d92b85::38b31716-8dbc-4c90-8aff-2b92e923a499',
  'doi': '10.23662/fors-ds-675-3'},
 {'id': '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/VLN5PZ',
  'doi': '10.34894/vln5pz'},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::dcb4c8fc-f69e-49d2-a7cb-48b9f7c007fa',
  'doi': '10.23662/fors-ds-686-1'},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::6e4b1efa-9e7d-43bb-86e5-8ffcfbca885d',
  'doi': '10.23662/fors-ds-690-1'},
 {'id': '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/JESW3L',
  'doi': '10.34894/jesw3l'},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::3557f168-dfca-4bc3-91b4-8d5ea39f902b',
  'doi': '10.23662/fors-ds-692-3'},
 {'id': '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/7LUSNC',
  'doi': '10.34894/7lusnc'}]

In [21]:
enriched_list = _ids_with_openaire_subjects(id_dois, doi_subjects)
enriched_list

[{'id': '8409b435-a7a8-4527-a955-436ad8d92b85::38b31716-8dbc-4c90-8aff-2b92e923a499',
  'doi': '10.23662/fors-ds-675-3',
  'enriched_subjects': ['8. Economic growth']},
 {'id': '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/VLN5PZ',
  'doi': '10.34894/vln5pz',
  'enriched_subjects': None},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::dcb4c8fc-f69e-49d2-a7cb-48b9f7c007fa',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': ['4. Education',
   '8. Economic growth',
   '10. No inequality',
   '3. Good health']},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::6e4b1efa-9e7d-43bb-86e5-8ffcfbca885d',
  'doi': '10.23662/fors-ds-690-1',
  'enriched_subjects': None},
 {'id': '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/JESW3L',
  'doi': '10.34894/jesw3l',
  'enriched_subjects': None},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::3557f168-dfca-4bc3-91b4-8d5ea39f902b',
  'doi': '10.23662/fors-ds-692-3',
  'enriched_subjects': None},
 {'id': '8393ec5c-7723-4f4e-9d63-f27fcad32

In [ ]:
# Only succesfully enriched IDs and subjects.
# Could be useful for comparing how many ids we sent to be enriched vs
# how many actually got new subjects
[el for el in enriched_list if el['enriched_subjects'] is not None]

[{'id': '8409b435-a7a8-4527-a955-436ad8d92b85::38b31716-8dbc-4c90-8aff-2b92e923a499',
  'doi': '10.23662/fors-ds-675-3',
  'enriched_subjects': ['8. Economic growth']},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::dcb4c8fc-f69e-49d2-a7cb-48b9f7c007fa',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': ['4. Education',
   '8. Economic growth',
   '10. No inequality',
   '3. Good health']}]

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword